# 步驟 4：Vectorbt 回測引擎

把推論出來的月度權重展開到日頻，用 **vectorbt** 模擬完整的每日淨值與最大回撤（Drawdown）。

## 回測設計

| 設定 | 值 | 說明 |
|------|-----|------|
| 調倉頻率 | 月底 | 每月最後交易日換倉 |
| 手續費 | 0.1425% | 雙邊，買入 1/3 計算方式 |
| 滑價 | 0.1% | 買入 +0.1%，賣出 -0.1% |
| 初始資金 | 1,000,000 | TWD |
| Benchmark | 0050 | 台灣 50 ETF |

## 注意：Regime Filter 效果
當多空濾網判斷為空頭時，該月所有倉位設為 0（全倉現金），避免在熊市硬撐倉位。

前置條件：已執行 `01` + `02` + `03`，變數 `weights_df`, `close`, `active_months`, `INIT_CASH`, `FEE` 存在記憶體中。

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

TEST_START = pd.Timestamp('2020-01-01')

# ── 月度權重 -> 日頻（往前填充）
weights_daily = weights_df.reindex(close.index, method='ffill').fillna(0.0)

# ── 對齊可用股票
close_clean = (
    close.loc[TEST_START:]
    .replace(0, np.nan)
    .ffill()
    .bfill()
    .dropna(axis=1)
)
common  = close_clean.columns.intersection(weights_daily.columns)
c_vbt   = close_clean[common]
w_vbt   = weights_daily.loc[TEST_START:, common]
bm_close = close_clean['0050'] if '0050' in close_clean.columns else close_clean.iloc[:, 0]

print(f'Universe  : {c_vbt.shape[1]} stocks x {c_vbt.shape[0]} days')
print(f'Running...')

In [ ]:
# ── 回測執行
pf = vbt.Portfolio.from_orders(
    close        = c_vbt,
    size         = w_vbt,
    size_type    = 'targetpercent',
    group_by     = True,
    cash_sharing = True,
    fees         = FEE,
    slippage     = SLIPPAGE,
    init_cash    = INIT_CASH,
)
pf_bm = vbt.Portfolio.from_holding(bm_close, init_cash=INIT_CASH)

# ── 計算指標
stats    = pf.stats()
stats_bm = pf_bm.stats()

eq     = pf.value()
eq_bm  = pf_bm.value()

n_years  = (eq.index[-1] - eq.index[0]).days / 365.25
cagr     = (eq.iloc[-1] / INIT_CASH) ** (1/n_years) - 1
cagr_bm  = (eq_bm.iloc[-1] / INIT_CASH) ** (1/n_years) - 1

monthly_eq  = eq.resample('ME').last()
monthly_ret = monthly_eq.pct_change().dropna()
sharpe      = (monthly_ret.mean() / monthly_ret.std()) * (12**0.5)

monthly_bm  = eq_bm.resample('ME').last().pct_change().dropna()
sharpe_bm   = (monthly_bm.mean() / monthly_bm.std()) * (12**0.5)

print()
print('  Strategy vs Benchmark (0050) -- vectorbt')
print('-' * 56)
print(f'  {"Metric":<22} {"Strategy":>12}   {"Benchmark (0050)":>16}')
print('-' * 56)

rows = [
    ('CAGR',                  f'{cagr:.2%}',                      f'{cagr_bm:.2%}'),
    ('Total Return',          f'{stats["Total Return [%]"]:.2f}%', f'{stats_bm["Total Return [%]"]:.2f}%'),
    ('Max Drawdown',          f'{stats["Max Drawdown [%]"]:.2f}%', f'{stats_bm["Max Drawdown [%]"]:.2f}%'),
    ('Sharpe (Monthly Ann.)', f'{sharpe:.2f}',                     f'{sharpe_bm:.2f}'),
    ('Active Months',         f'{active_months} / {len(weights_df)}', 'N/A'),
]
for label, s_val, b_val in rows:
    print(f'  {label:<22} {s_val:>12}   {b_val:>16}')
print('-' * 56)

In [ ]:
# ── Equity Curve 視覺化
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=eq.index, y=eq.values / INIT_CASH * 100 - 100,
    name='Strategy', line=dict(color='#2962FF', width=2)
))
fig.add_trace(go.Scatter(
    x=eq_bm.index, y=eq_bm.values / INIT_CASH * 100 - 100,
    name='0050 Benchmark', line=dict(color='#888888', width=1.5, dash='dot')
))
fig.add_hline(y=0, line_color='#444444', line_dash='dash', line_width=0.8)
fig.update_layout(
    title='Cumulative Return: Strategy vs Benchmark',
    xaxis_title='Date',
    yaxis_title='Return (%)',
    template='plotly_dark',
    height=450,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

In [ ]:
# ── 月度報酬熱力圖
monthly_pivot = (
    monthly_ret
    .to_frame('ret')
    .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
    .pivot(index='year', columns='month', values='ret') * 100
)
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec']

fig2 = go.Figure(go.Heatmap(
    z=monthly_pivot.values,
    x=monthly_pivot.columns.tolist(),
    y=monthly_pivot.index.tolist(),
    colorscale='RdYlGn',
    zmid=0,
    text=monthly_pivot.round(1).values,
    texttemplate='%{text}%',
    showscale=True,
))
fig2.update_layout(
    title='Monthly Returns Heatmap (%)',
    template='plotly_dark',
    height=380,
)
fig2.show()

In [ ]:
# ── 儲存結果（選用：若要用於 reports/generate_report.py）
import pickle

eq.to_pickle('../eq.pkl')
eq_bm.to_pickle('../bm_eq.pkl')
weights_df.to_pickle('../weights.pkl')
final_preds.to_pickle('../predictions.pkl')

print('Saved: eq.pkl / bm_eq.pkl / weights.pkl / predictions.pkl')
print(f'Total Return: {cagr:.2%} CAGR  |  Sharpe: {sharpe:.2f}  |  Max DD: {stats["Max Drawdown [%]"]:.2f}%')